# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR^2 dataset using the `mlcroissant` library. We focus on loading, reviewing, and preparing the dataset described by its Croissant schema with all references to dataset entities such as record sets and fields by their unique `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print metadata (access as attributes, not via indexing)
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Citation: {dataset.metadata.citeAs}")


## 2. Data Overview

Explore available **record sets** in the dataset by their `@id` and inspect the fields and columns within each record set.

All references are by `@id`.

In [ ]:
# List available record sets (entities with type 'RecordSet')
print("Available Record Sets (by @id):\n-----------------------------")
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")

# For each record set, list fields and columns
print("\nFields and columns for each record set:")
overview = {}
for rs in dataset.metadata.record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            col_ids = getattr(field, 'columns', []) or []
            print(f"    - @id: {field.id} | name: {getattr(field, 'name', field.id)} | Columns: {[c.id for c in col_ids] if col_ids else '[]'}")
            overview.setdefault(rs.id, []).append({'field_id': field.id, 'columns': [c.id for c in col_ids] if col_ids else []})
    else:
        print("   No fields present.")

## 3. Data Extraction

Load the record sets as DataFrames for data analysis. We provide a list of all available record set `@id`s collected above.

In [ ]:
# Collect record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # Load all records for each set
    print(f"Loading records from {rs_id}...")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns for {rs_id}: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print(f"No records found for {rs_id}.")

## 4. Exploratory Data Analysis (EDA)

Now, let's select a record set and numeric field for EDA.

> You may need to adjust the `selected_record_set_id` and `numeric_field_id` according to the outputs above. Below is an example using the first loaded record set (if any).

In [ ]:
# Example: pick the first available record set and numeric field
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    print(f"Using {selected_record_set_id} with columns: {df.columns.tolist()}")
    
    # Try to find a likely numeric field (for demonstration, use 'cr:mw' if present)
    numeric_field_id = None
    for col in df.columns:
        # Try standard numeric/protein fields
        if col.lower() in ['cr:mw', 'cr:coverage', 'cr:peptides', 'cr:score'] or pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if not numeric_field_id:
        # Fallback: choose the first column
        numeric_field_id = df.columns[0]
    print(f"Selected numeric field: {numeric_field_id}")
    
    # Set threshold for filtering
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    try:
        filtered_df = df[df[numeric_field_id] > threshold]
    except Exception as e:
        print(f"Filtering failed: {e}")
        filtered_df = df
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    
    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    try:
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized values for {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
    except Exception as e:
        print(f"Normalization failed: {e}")
    
    # Group by a likely categorical field (e.g., 'cr:description' or others)
    potential_groups = [col for col in df.columns if col != numeric_field_id]
    group_field = None
    for col in potential_groups:
        if df[col].dtype == 'O' and df[col].nunique() < 20:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped means by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No dataframes extracted in the prior step.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its normalized form.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and (numeric_field_id in df.columns):
    plt.figure(figsize=(12, 5))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    if f"{numeric_field_id}_normalized" in df.columns:
        plt.subplot(1,2,2)
        sns.histplot(df[f"{numeric_field_id}_normalized"].dropna(), kde=True, bins=30, color='coral')
        plt.title(f"Normalized {numeric_field_id}")
        plt.xlabel(f"{numeric_field_id}_normalized")
    plt.tight_layout()
    plt.show()
else:
    print("Visualization skipped: No numeric data available.")

## 6. Conclusion

- We loaded and explored the FAIR^2 dataset's record sets and fields using `mlcroissant` with all references by `@id`.
- Dataframes for all record sets were loaded and inspected for numeric analysis fields.
- Basic EDA steps (filtering and normalization) and visualization were demonstrated on a selected numeric field.

You may further adapt this notebook for advanced analysis or integrate with your ML workflow using the provided dataframes.